In [0]:
df_silver_date = spark.read.table("he_prod_incen_analytics_dbw_01.silver.dim_date")

In [0]:
from pyspark.sql.functions import date_format, year, month, col, upper

# 1. Create Surrogate Key: FORMAT DATE to 'YYYYMM' as INT
# 2. Extract month_name (JANUARY, FEBRUARY) -> Standardize to UPPER
# 3. Extract year_num

df_dim_month = df_silver_date.select(
    date_format(col("date"), "yyyyMM").cast("int").alias("month_sk"),
    upper(date_format(col("date"), "MMMM")).alias("month_name"),
    year(col("date")).alias("year_num")
)

# 3. Deduplicate (Keep only one row per month)
df_dim_month = df_dim_month.dropDuplicates(["month_sk"])

display(df_dim_month)

In [0]:
gold_table_fqn = "he_prod_incen_analytics_dbw_01.gold.dim_month"

df_dim_month.write.mode("overwrite") \
  .option("overwriteSchema", "true") \
  .saveAsTable(gold_table_fqn)

print(f"✅ Successfully wrote Dim_Month to Gold layer.")